# Phase 3 database loading and query execution
# 1. Connect to Oracle
# 2. Load cleaned CSV datasets
# 3. Insert rows into database tables
# 4. Execute SQL queries used in the report

In [1]:
## Step 1: Connect to Oracle database
## We connect to the UBC Oracle server using the oracledb Python library.
## External libraries used: oracledb

import oracledb

dsn = oracledb.makedsn("localhost", 1522, service_name="stu")

connection = oracledb.connect(
    user="ora_timzeng",
    password="a89468680",
    dsn=dsn
)

cursor = connection.cursor()

print("Connected")

Connected


In [2]:
with open("create_tables.sql", "r") as f:
    sql_script = f.read()

for statement in sql_script.split(";"):
    if statement.strip():
        try:
            cursor.execute(statement)
            print("OK")
        except Exception as e:
            msg = str(e)
            if "ORA-00942" in msg:
                print("SKIPPED missing table")
            else:
                print("FAILED")
                print(statement)
                print(e)
                break

connection.commit()
print("Schema ready")

OK
OK
OK
OK
FAILED

DROP TABLE IMDB_BASICS PURGE
ORA-02449: unique/primary keys in table referenced by foreign keys
Help: https://docs.oracle.com/error-help/db/ora-02449/
Schema ready


In [3]:
## Step 2: Create database schema
## The schema is defined in create_tables.sql and executed from Python.

cursor.execute("SELECT table_name FROM user_tables ORDER BY table_name")
for row in cursor:
    print(row[0])

AUTHORS
EDITORS
IMDB_BASICS
MOVIES
PUBLISHERS
SALES
SALESDETAILS
TITLEAUTHORS
TITLEDITORS
TITLES
TMDB_METADATA


In [10]:
## Step 3: Load cleaned datasets
## Each dataset is inserted using executemany() for efficiency.

import csv

rows = []

with open("data files/movies.csv", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for r in reader:
        rows.append((
            int(r["movieId"]),
            r["title"],
            r["genres"]
        ))

cursor.executemany(
    "INSERT INTO MOVIES (movieId, title, genres) VALUES (:1,:2,:3)",
    rows
)

connection.commit()

print("Loaded MOVIES:", len(rows))

IntegrityError: ORA-00001: unique constraint (ORA_TIMZENG.SYS_C002747481) violated
Help: https://docs.oracle.com/error-help/db/ora-00001/

In [11]:
rows = []

with open("data files/imdb_basics.csv", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for r in reader:
        rows.append((
            r["tconst"],
            r["imdb_title"],
            int(float(r["start_year"])) if r["start_year"] else None,
            r["imdb_genres"]
        ))

cursor.executemany(
    "INSERT INTO IMDB_BASICS (tconst, imdb_title, start_year, imdb_genres) VALUES (:1,:2,:3,:4)",
    rows
)

connection.commit()

print("Loaded IMDB_BASICS:", len(rows))

IntegrityError: ORA-00001: unique constraint (ORA_TIMZENG.SYS_C002747482) violated
Help: https://docs.oracle.com/error-help/db/ora-00001/

In [12]:
rows = []

with open("data files/imdb_ratings.csv", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for r in reader:
        rows.append((
            r["tconst"],
            float(r["imdb_avg_rating"]) if r["imdb_avg_rating"] else None,
            int(float(r["imdb_num_votes"])) if r["imdb_num_votes"] else None
        ))

cursor.executemany(
    "INSERT INTO IMDB_RATINGS (tconst, imdb_avg_rating, imdb_num_votes) VALUES (:1,:2,:3)",
    rows
)

connection.commit()

print("Loaded IMDB_RATINGS:", len(rows))

Loaded IMDB_RATINGS: 1000


In [13]:
rows = []

with open("data files/tmdb_metadata.csv", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for r in reader:
        rows.append((
            int(float(r["tmdb_id"])) if r["tmdb_id"] else None,
            r["imdb_id"],
            r["tmdb_title"],
            int(float(r["release_year"])) if r["release_year"] else None,
            r["tmdb_genres"],
            float(r["tmdb_vote_average"]) if r["tmdb_vote_average"] else None,
            int(float(r["tmdb_vote_count"])) if r["tmdb_vote_count"] else None
        ))

cursor.executemany(
    """
    INSERT INTO TMDB_METADATA
    (tmdb_id, imdb_id, tmdb_title, release_year, tmdb_genres, tmdb_vote_average, tmdb_vote_count)
    VALUES (:1,:2,:3,:4,:5,:6,:7)
    """,
    rows
)

connection.commit()

print("Loaded TMDB_METADATA:", len(rows))

IntegrityError: ORA-00001: unique constraint (ORA_TIMZENG.SYS_C002747485) violated
Help: https://docs.oracle.com/error-help/db/ora-00001/

In [14]:
rows = []

with open("data files/links.csv", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for r in reader:
        rows.append((
            int(r["movieId"]),
            r["tconst"],                          
            int(float(r["tmdbId"])) if r["tmdbId"] else None
        ))

cursor.executemany(
    "INSERT INTO LINKS (movieId, tconst, tmdbId) VALUES (:1,:2,:3)",
    rows
)

connection.commit()

print("Loaded LINKS:", len(rows))

Loaded LINKS: 1000


In [15]:
from collections import defaultdict

ratings_sum = defaultdict(float)
ratings_count = defaultdict(int)

with open("data files/ML_ratings.csv", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for r in reader:
        movie = int(r["movieId"])
        rating = float(r["rating"])
        
        ratings_sum[movie] += rating
        ratings_count[movie] += 1

rows = []
for movie in ratings_sum:
    avg = ratings_sum[movie] / ratings_count[movie]
    rows.append((movie, avg))

cursor.executemany(
    "INSERT INTO ML_RATINGS (movieId, rating) VALUES (:1,:2)",
    rows
)

connection.commit()

print("Loaded ML_RATINGS:", len(rows))

Loaded ML_RATINGS: 991


In [16]:
cursor.execute("DELETE FROM ML_TAGS")
connection.commit()

rows_set = set()

with open("data files/ML_tags.csv", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for r in reader:
        rows_set.add((
            int(r["movieId"]),
            r["tag"]
        ))

rows = list(rows_set)

cursor.executemany(
    "INSERT INTO ML_TAGS (movieId, tag) VALUES (:1,:2)",
    rows
)

connection.commit()

print("Loaded ML_TAGS:", len(rows))

Loaded ML_TAGS: 2601


In [17]:
## Step 4: Validate database contents
## We verify row counts for each table.

tables = [
    "MOVIES",
    "IMDB_BASICS",
    "IMDB_RATINGS",
    "TMDB_METADATA",
    "LINKS",
    "ML_RATINGS",
    "ML_TAGS"
]

for t in tables:
    cursor.execute(f"SELECT COUNT(*) FROM {t}")
    count = cursor.fetchone()[0]
    print(t, ":", count)

MOVIES : 1000
IMDB_BASICS : 1000
IMDB_RATINGS : 1000
TMDB_METADATA : 1000
LINKS : 1000
ML_RATINGS : 991
ML_TAGS : 2601


In [22]:
## Step 5: Research queries
## These queries answer the project research questions

## Main Question: How consistent are movie ratings between IMDb and 
## MovieLens for the same movies released between 2020 and 2025?

cursor.execute("""
SELECT m.title,
       r.rating AS movielens_rating,
       ir.imdb_avg_rating,
       ABS(ir.imdb_avg_rating - r.rating) AS disagreement
FROM MOVIES m
JOIN ML_RATINGS r ON m.movieId = r.movieId
JOIN LINKS l ON m.movieId = l.movieId
JOIN IMDB_RATINGS ir ON l.tconst = ir.tconst
FETCH FIRST 10 ROWS ONLY
""")

for row in cursor:
    print(row)

('Eyimofe (This Is My Desire) (2020)', 3.6666666666666665, 7, 3.3333333333333335)
('Break Even (2020)', 3.0, 4.7, 1.7)
('Faith Under Fire (2020)', 3.75, 3.4, 0.35)
('The Family Tree (2020)', 2.0, 4.8, 2.8)
('Suraj Pe Mangal Bhari (2020)', 2.5, 6, 3.5)
('Cowboys (2020)', 3.25, 6.4, 3.15)
('The Killing of Two Lovers (2021)', 3.4696969696969697, 6.8, 3.33030303030303)
('Green Sea (2020)', 4.0, 6.5, 2.5)
('Paper Spiders (2020)', 3.5, 6.7, 3.2)
('Reversion (2020)', 2.0, 3.5, 1.5)


In [24]:
## Sub Question 1: Does the average disagreement (mean |Alignment|) 
## differ across selected genres (2020–2025)?

cursor.execute("""
SELECT m.genres,
       AVG(ABS(ir.imdb_avg_rating - r.rating)) AS avg_disagreement
FROM MOVIES m
JOIN ML_RATINGS r ON m.movieId = r.movieId
JOIN LINKS l ON m.movieId = l.movieId
JOIN IMDB_RATINGS ir ON l.tconst = ir.tconst
GROUP BY m.genres
ORDER BY avg_disagreement DESC
FETCH FIRST 10 ROWS ONLY
""")

for row in cursor:
    print(row)

('Comedy|Drama|Thriller', 5.233333333333333)
('Action|Horror|Thriller', 4.8)
('Action|Comedy|Thriller', 4.4)
('Mystery', 4.1)
('Action|Drama', 4.057440476190476)
('Comedy|Documentary', 4)
('Crime|Drama', 3.9833333333333334)
('Drama|Fantasy', 3.966666666666667)
('Comedy|Fantasy|Sci-Fi', 3.9)
('Children|Romance', 3.8333333333333335)


In [35]:
## Sub question 2: Is there a measurable correlation between user engagement 
## (MovieLens tag counts and IMDb vote counts) and disagreement magnitude?

cursor.execute("""
SELECT m.title,
       COUNT(DISTINCT t.tag) AS tag_count,
       ir.imdb_num_votes,
       ABS(ir.imdb_avg_rating - AVG(r.rating)) AS disagreement
FROM MOVIES m
JOIN LINKS l ON m.movieId = l.movieId
JOIN IMDB_RATINGS ir ON l.tconst = ir.tconst
JOIN ML_RATINGS r ON m.movieId = r.movieId
LEFT JOIN ML_TAGS t ON m.movieId = t.movieId
GROUP BY m.movieId, m.title, ir.imdb_num_votes, ir.imdb_avg_rating
HAVING COUNT(DISTINCT t.tag) > 0
ORDER BY disagreement ASC
FETCH FIRST 10 ROWS ONLY
""")

for row in cursor:
    print(row)

('War of Likes (2021)', 0, 606, 0)
('Miss India (2020)', 0, 3481, 0.1)
('Cream of the Crop (2022)', 0, 163, 0.1)
('Being Dead (2021)', 0, 149, 0.2)
('Plaza (2020)', 0, 307, 0.2)
('Les Gagnants (2022)', 0, 153, 0.2)
('Kitty Mammas (2020)', 1, 187, 0.2)
('One Life at a Time (2020)', 0, 93, 0.3)
('Faith Under Fire (2020)', 0, 492, 0.35)
('Fast and Fierce: Death Race (2020)', 0, 637, 0.4)


## Database loading completed

Tables loaded:

MOVIES - 1000  
IMDB_BASICS – 1000  
IMDB_RATINGS – 1000  
TMDB_METADATA – 1000  
LINKS – 1000  
ML_RATINGS – 991  
ML_TAGS – 2601  

These tables support cross-platform analysis between MovieLens, IMDB, and TMDB datasets.

In [ ]:
cursor.close()
connection.close()
print("Connection closed")